Step 1 Import the dataset using Python and Pandas

In [1]:
import pandas as pd 
#load the raw netflix dataset
df = pd.read_csv("../data/raw/Dataset.csv")
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


Step 2: Identify and handle missing values

In [2]:
#check that the file loaded correctly and matches what we expect
df.shape
#.info() shows column names, non-null counts, and data types
df.info()
#.isnull() only catches real NaN cells
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

In [3]:
#since .isnull() cant see "Not Given" we count it manually here
(df == "Not Given").sum()

show_id            0
type               0
title              0
director        2588
country          287
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
dtype: int64

In [4]:
#Replace the "Not Given" placeholder with "Unknown" in both columns.
# We fill rather than drop because:
#   - director is missing in ~29% of rows - dropping would lose too much data
#   - country is missing in only ~3%, but there's no analytical benefit
df['director'] = df['director'].replace("Not Given", "Unknown")
df['country'] = df['country'].replace("Not Given", "Unknown")

In [5]:
#should now show 0 for both columns
(df == "Not Given").sum()

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

Step 3: Remove duplicate records and formatting
inconsistencies

In [6]:
#check for fully duplicate rows first
df.duplicated().sum()

np.int64(0)

In [7]:
df.duplicated(subset='show_id').sum()

np.int64(0)

In [8]:
df['type'].unique()

<StringArray>
['Movie', 'TV Show']
Length: 2, dtype: str

In [9]:
df['rating'].unique()

<StringArray>
[   'PG-13',    'TV-MA',    'TV-PG',    'TV-14',    'TV-Y7',     'TV-Y',
       'PG',     'TV-G',        'R',        'G',    'NC-17',       'NR',
 'TV-Y7-FV',       'UR']
Length: 14, dtype: str

In [11]:
text_cols = df.select_dtypes(include='str').columns
for col in text_cols:
    df[col] = df[col].str.strip()

In [12]:
for col in df.select_dtypes(include='str').columns:
    has_whitespace = (df[col] != df[col].str.strip()).sum()
    if has_whitespace > 0:
        print(col, "has", has_whitespace, "values with extra whitespace")